<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">처음부터 만드는 대형 언어 모델</a> 책의 보조 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 2장: 텍스트 데이터 다루기

이 노트북에서 사용되는 패키지들:

In [ ]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

- 이 장에서는 LLM을 위한 입력 데이터를 "준비"하기 위한 데이터 전처리와 샘플링을 다룹니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/01.webp?timestamp=1" width="500px">

## 2.1 단어 임베딩 이해하기

- 이 섹션에는 코드가 없습니다

- 임베딩에는 여러 형태가 있습니다; 이 책에서는 텍스트 임베딩에 초점을 맞춥니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/02.webp" width="500px">

- LLM은 고차원 공간의 임베딩을 사용합니다 (즉, 수천 개의 차원)
- 이러한 고차원 공간을 시각화할 수 없으므로 (인간은 1, 2, 또는 3차원으로 생각합니다), 아래 그림은 2차원 임베딩 공간을 보여줍니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/03.webp" width="300px">

## 2.2 텍스트 토큰화

- 이 섹션에서는 텍스트를 토큰화합니다. 이는 텍스트를 개별 단어와 구두점 문자와 같은 더 작은 단위로 나누는 것을 의미합니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/04.webp" width="300px">

- 작업할 원시 텍스트를 로드합니다
- [Edith Wharton의 The Verdict](https://en.wikisource.org/wiki/The_Verdict)는 공개 도메인 단편 소설입니다

In [ ]:
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

- (이전 코드 셀을 실행할 때 `ssl.SSLCertVerificationError`가 발생하면, 오래된 Python 버전을 사용하고 있기 때문일 수 있습니다; [GitHub에서 더 자세한 정보를 확인할 수 있습니다](https://github.com/rasbt/LLMs-from-scratch/pull/403))

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

- 목표는 LLM을 위해 이 텍스트를 토큰화하고 임베딩하는 것입니다
- 먼저 간단한 샘플 텍스트를 기반으로 간단한 토크나이저를 개발한 후, 위의 텍스트에 적용해보겠습니다
- 다음 정규식은 공백으로 분할합니다

In [ ]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

- 공백뿐만 아니라 쉼표와 마침표로도 분할하고 싶으므로, 정규식을 수정해보겠습니다

In [ ]:
result = re.split(r'([,.]|\s)', text)

print(result)

- 보시다시피, 이는 빈 문자열을 생성합니다. 이를 제거해보겠습니다

In [ ]:
# 각 항목에서 공백을 제거한 후 빈 문자열을 필터링합니다.
result = [item for item in result if item.strip()]
print(result)

- 꽤 좋아 보입니다. 하지만 마침표, 물음표 등 다른 유형의 구두점도 처리해보겠습니다

In [ ]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

- 이는 꽤 좋습니다. 이제 이 토큰화를 원시 텍스트에 적용할 준비가 되었습니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/05.webp" width="350px">

In [ ]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

- 전체 토큰 수를 계산해보겠습니다

In [ ]:
print(len(preprocessed))

## 2.3 토큰을 토큰 ID로 변환하기

- 다음으로, 나중에 임베딩 레이어를 통해 처리할 수 있는 토큰 ID로 텍스트 토큰을 변환합니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/06.webp" width="500px">

- 이러한 토큰들로부터 모든 고유 토큰으로 구성된 어휘를 구축할 수 있습니다

In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

In [ ]:
vocab = {token:integer for integer,token in enumerate(all_words)}

- 다음은 이 어휘의 첫 50개 항목입니다:

In [ ]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

- 아래에서는 작은 어휘를 사용하여 짧은 샘플 텍스트의 토큰화를 설명합니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/07.webp?123" width="500px">

- 이제 모든 것을 토크나이저 클래스로 통합합니다

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # 지정된 구두점 앞의 공백을 제거합니다
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

- `encode` 함수는 텍스트를 토큰 ID로 변환합니다
- `decode` 함수는 토큰 ID를 다시 텍스트로 변환합니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/08.webp?123" width="500px">

- 토크나이저를 사용하여 텍스트를 정수로 인코딩(즉, 토큰화)할 수 있습니다
- 이러한 정수들은 나중에 LLM의 입력으로 임베딩될 수 있습니다

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

- 정수를 다시 텍스트로 디코딩할 수 있습니다

In [ ]:
tokenizer.decode(ids)

In [ ]:
tokenizer.decode(tokenizer.encode(text))

## 2.4 특수 컨텍스트 토큰 추가하기

- 알려지지 않은 단어와 텍스트의 끝을 나타내기 위해 일부 "특수" 토큰을 추가하는 것이 유용합니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/09.webp?123" width="500px">

- 일부 토크나이저는 특수 토큰을 사용하여 LLM에 추가 컨텍스트를 제공합니다
- 이러한 특수 토큰 중 일부는 다음과 같습니다
  - `[BOS]` (beginning of sequence) 텍스트의 시작을 표시합니다
  - `[EOS]` (end of sequence) 텍스트가 끝나는 위치를 표시합니다 (일반적으로 서로 관련 없는 여러 텍스트를 연결할 때 사용됩니다. 예: 두 개의 다른 위키백과 기사나 두 권의 다른 책 등)
  - `[PAD]` (padding) 배치 크기가 1보다 큰 상태로 LLM을 훈련하는 경우 (서로 다른 길이의 여러 텍스트를 포함할 수 있습니다; 패딩 토큰으로 짧은 텍스트를 가장 긴 길이로 맞춰 모든 텍스트가 동일한 길이를 갖도록 합니다)
- `[UNK]` 어휘에 포함되지 않은 단어를 나타냅니다

- GPT-2는 위에서 언급한 토큰들을 필요로 하지 않으며, 복잡성을 줄이기 위해 `<|endoftext|>` 토큰만 사용합니다
- `<|endoftext|>`는 위에서 언급한 `[EOS]` 토큰과 유사합니다
- GPT도 패딩을 위해 `<|endoftext|>`를 사용합니다 (배치된 입력으로 훈련할 때 일반적으로 마스크를 사용하므로 패딩된 토큰에는 어차피 주의를 기울이지 않으므로, 이러한 토큰이 무엇인지는 중요하지 않습니다)
- GPT-2는 어휘에 없는 단어에 대해 `<UNK>` 토큰을 사용하지 않습니다; 대신 GPT-2는 단어를 하위 단어 단위로 분해하는 바이트 쌍 인코딩(BPE) 토크나이저를 사용하며, 이에 대해서는 이후 섹션에서 논의하겠습니다


- 두 개의 독립적인 텍스트 소스 사이에 `<|endoftext|>` 토큰을 사용합니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/10.webp" width="500px">

- 다음 텍스트를 토큰화하면 어떻게 되는지 살펴보겠습니다:

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)

- 위의 코드는 "Hello"라는 단어가 어휘에 포함되어 있지 않기 때문에 오류를 발생시킵니다
- 이러한 경우를 처리하기 위해 알려지지 않은 단어를 나타내는 `"<|unk|>"`와 같은 특수 토큰을 어휘에 추가할 수 있습니다
- 이미 어휘를 확장하고 있으므로, GPT-2 훈련에서 텍스트의 끝을 나타내는 데 사용되는 `"<|endoftext|>"`라는 또 다른 토큰을 추가해보겠습니다 (또한 연결된 텍스트 사이에도 사용됩니다. 예를 들어, 훈련 데이터셋이 여러 기사, 책 등으로 구성되어 있는 경우)

In [ ]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [ ]:
len(vocab.items())

In [ ]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

- 또한 새로운 `<unk>` 토큰을 언제, 어떻게 사용해야 하는지 알 수 있도록 토크나이저를 적절히 조정해야 합니다

In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # 지정된 구두점 앞의 공백을 제거합니다
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

수정된 토크나이저로 텍스트를 토큰화해보겠습니다:

In [ ]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

In [ ]:
tokenizer.encode(text)

In [ ]:
tokenizer.decode(tokenizer.encode(text))

## 2.5 바이트 쌍 인코딩

- GPT-2는 토크나이저로 바이트 쌍 인코딩(BPE)을 사용했습니다
- 이를 통해 모델은 미리 정의된 어휘에 없는 단어를 더 작은 하위 단어 단위 또는 개별 문자로 분해할 수 있어, 어휘에 없는 단어를 처리할 수 있습니다
- 예를 들어, GPT-2의 어휘에 "unfamiliarword"라는 단어가 없다면, 훈련된 BPE 병합에 따라 ["unfam", "iliar", "word"] 또는 다른 하위 단어 분해로 토큰화할 수 있습니다
- 원래 BPE 토크나이저는 여기에서 찾을 수 있습니다: [https://github.com/openai/gpt-2/blob/master/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- 이 장에서는 OpenAI의 오픈소스 [tiktoken](https://github.com/openai/tiktoken) 라이브러리의 BPE 토크나이저를 사용합니다. 이는 계산 성능을 향상시키기 위해 핵심 알고리즘을 Rust로 구현했습니다
- [./bytepair_encoder](../02_bonus_bytepair-encoder)에서 이 두 구현을 나란히 비교하는 노트북을 만들었습니다 (tiktoken이 샘플 텍스트에서 약 5배 빨랐습니다)

In [ ]:
# pip install tiktoken

In [ ]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

In [ ]:
strings = tokenizer.decode(integers)

print(strings)

- BPE 토크나이저는 알려지지 않은 단어를 하위 단어와 개별 문자로 분해합니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/11.webp" width="300px">

## 2.6 슬라이딩 윈도우를 사용한 데이터 샘플링

- LLM은 한 번에 한 단어씩 생성하도록 훈련하므로, 시퀀스의 다음 단어가 예측할 대상을 나타내도록 훈련 데이터를 준비하려고 합니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/12.webp" width="400px">

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

- 각 텍스트 청크에 대해 입력과 타겟이 필요합니다
- 모델이 다음 단어를 예측하기를 원하므로, 타겟은 입력을 오른쪽으로 한 위치씩 이동한 것입니다

In [ ]:
enc_sample = enc_text[50:]

In [ ]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

- 하나씩, 예측은 다음과 같이 보일 것입니다:

In [ ]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

In [ ]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

- 어텐션 메커니즘을 다룬 후 이후 장에서 다음 단어 예측을 처리하겠습니다
- 지금은 입력 데이터셋을 반복하고 하나씩 이동된 입력과 타겟을 반환하는 간단한 데이터 로더를 구현합니다

- PyTorch 설치 및 가져오기 (설치 팁은 부록 A 참조)

In [ ]:
import torch
print("PyTorch version:", torch.__version__)

- 슬라이딩 윈도우 접근 방식을 사용하여 위치를 +1씩 변경합니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/13.webp?123" width="500px">

- 입력 텍스트 데이터셋에서 청크를 추출하는 데이터셋과 데이터로더를 생성합니다

In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # 전체 텍스트를 토큰화합니다
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # 슬라이딩 윈도우를 사용하여 책을 max_length의 겹치는 시퀀스로 청킹합니다
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # 토크나이저를 초기화합니다
    tokenizer = tiktoken.get_encoding("gpt2")

    # 데이터셋을 생성합니다
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # 데이터로더를 생성합니다
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

- 컨텍스트 크기가 4인 LLM에 대해 배치 크기 1로 데이터로더를 테스트해보겠습니다:

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

In [ ]:
second_batch = next(data_iter)
print(second_batch)

- 아래와 같이 스트라이드가 컨텍스트 길이(여기서는 4)와 동일한 예시입니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/14.webp" width="500px">

- 배치된 출력도 생성할 수 있습니다
- 여기서 스트라이드를 증가시켜 배치 간에 겹침이 없도록 하는데, 더 많은 겹침은 과적합을 증가시킬 수 있기 때문입니다

In [ ]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

## 2.7 토큰 임베딩 생성하기

- 데이터는 이미 LLM을 위해 거의 준비되었습니다
- 하지만 마지막으로 임베딩 레이어를 사용하여 토큰을 연속적인 벡터 표현으로 임베딩해보겠습니다
- 일반적으로 이러한 임베딩 레이어는 LLM 자체의 일부이며 모델 훈련 중에 업데이트(훈련)됩니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/15.webp" width="400px">

- 토큰화 후 입력 ID가 2, 3, 5, 1인 다음 4개의 입력 예시가 있다고 가정해보겠습니다:

In [ ]:
input_ids = torch.tensor([2, 3, 5, 1])

- 단순화를 위해 6개 단어만의 작은 어휘를 가지고 크기 3의 임베딩을 생성한다고 가정해보겠습니다:

In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- 이는 6x3 가중치 행렬을 생성합니다:

In [ ]:
print(embedding_layer.weight)

- 원-핫 인코딩에 익숙한 분들에게는, 위의 임베딩 레이어 접근 방식이 본질적으로 완전 연결 레이어에서 원-핫 인코딩 후 행렬 곱셈을 구현하는 더 효율적인 방법일 뿐이며, 이는 [./embedding_vs_matmul](../03_bonus_embedding-vs-matmul)의 보조 코드에 설명되어 있습니다
- 임베딩 레이어는 원-핫 인코딩과 행렬 곱셈 접근 방식과 동등한 더 효율적인 구현일 뿐이므로, 역전파를 통해 최적화할 수 있는 신경망 레이어로 볼 수 있습니다

- ID가 3인 토큰을 3차원 벡터로 변환하려면 다음과 같이 합니다:

In [ ]:
print(embedding_layer(torch.tensor([3])))

- 위의 결과는 `embedding_layer` 가중치 행렬의 4번째 행입니다
- 위의 4개 `input_ids` 값을 모두 임베딩하려면 다음과 같이 합니다

In [ ]:
print(embedding_layer(input_ids))

- 임베딩 레이어는 본질적으로 조회 연산입니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/16.webp?123" width="500px">

- **임베딩 레이어와 일반적인 선형 레이어를 비교하는 보너스 콘텐츠에 관심이 있으실 수 있습니다: [../03_bonus_embedding-vs-matmul](../03_bonus_embedding-vs-matmul)**

## 2.8 단어 위치 인코딩

- 임베딩 레이어는 입력 시퀀스에서의 위치에 관계없이 ID를 동일한 벡터 표현으로 변환합니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/17.webp" width="400px">

- 위치 임베딩은 토큰 임베딩 벡터와 결합되어 대형 언어 모델을 위한 입력 임베딩을 형성합니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/18.webp" width="500px">

- 바이트 쌍 인코더는 50,257의 어휘 크기를 가집니다:
- 입력 토큰을 256차원 벡터 표현으로 인코딩하려고 한다고 가정해보겠습니다:

In [ ]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- 데이터로더에서 데이터를 샘플링하면, 각 배치의 토큰을 256차원 벡터로 임베딩합니다
- 각각 4개의 토큰을 가진 배치 크기 8이 있다면, 이는 8 x 4 x 256 텐서를 생성합니다:

In [ ]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [ ]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

In [ ]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# 임베딩이 어떻게 생겼는지 보려면 다음 줄의 주석을 해제하고 실행하세요
# print(token_embeddings)

- GPT-2는 절대 위치 임베딩을 사용하므로, 다른 임베딩 레이어를 생성하기만 하면 됩니다:

In [ ]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# 임베딩 레이어 가중치가 어떻게 생겼는지 보려면 다음 줄의 주석을 해제하고 실행하세요
# print(pos_embedding_layer.weight)

In [ ]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

# 임베딩이 어떻게 생겼는지 보려면 다음 줄의 주석을 해제하고 실행하세요
# print(pos_embeddings)

- LLM에서 사용되는 입력 임베딩을 생성하기 위해, 토큰과 위치 임베딩을 단순히 더합니다:

In [ ]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

# 임베딩이 어떻게 생겼는지 보려면 다음 줄의 주석을 해제하고 실행하세요
# print(input_embeddings)

- 입력 처리 워크플로의 초기 단계에서 입력 텍스트가 별도의 토큰으로 분할됩니다
- 이 분할에 이어, 이러한 토큰들은 미리 정의된 어휘를 기반으로 토큰 ID로 변환됩니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/19.webp" width="400px">

# 요약 및 핵심사항

이 장에서 구현한 데이터 로더의 간결한 버전이며 앞으로 장에서 GPT 모델을 훈련하는 데 필요한 [./dataloader.ipynb](./dataloader.ipynb) 코드 노트북을 참조하세요.

연습 문제 해답은 [./exercise-solutions.ipynb](./exercise-solutions.ipynb)를 참조하세요.

GPT-2 토크나이저를 처음부터 구현하고 훈련하는 방법을 배우는 데 관심이 있다면 [바이트 쌍 인코딩(BPE) 토크나이저 처음부터 만들기](../02_bonus_bytepair-encoder/compare-bpe-tiktoken.ipynb) 노트북을 참조하세요.